### PENGUIN (Neshyba & Eklof, 2026)

## Project template

This notebook is meant to help you get started on your project by providing examples of how to implement various project ideas. Some of the meclib code library functions will look familiar, but there's also a new one that you might find handy.

## Learning goals
1. I'm familiar with how to use the following functions:
- cl.CreateClimateParams
- cl.LoadMyScenario
- cl.run_Cambio
- cl.CS_list_plots
- cl.CS_list_compare
2. I can modify a climate parameter dictionary (e.g., by using commands like "ClimateParams_biggerTstar['albedo_Tstar']=2.5") 

In [ ]:
import numpy as np
from pint import UnitRegistry; AssignQuantity = UnitRegistry().Quantity
import pandas as pd
import matplotlib.pyplot as plt; plt.rc("figure", figsize=(12,8))
import meclib.cl as cl
from copy import copy as makeacopy

### Loading in a pre-calculated emission scenario
You can use the scenario already loaded into this space, or upload your own.

In [ ]:
filename = 'RCP4_5.pkl'
time, eps, epsdictionary_from_file = cl.LoadMyScenario(filename,verbose=True)

### Modifying a climate parameter dictionary
The function **cl.CreateClimateParams** creates a climate parameter dictionary, which will be required later when you want to run a Cambio model. The function requires as input the dictionary from the emission scenario you just loaded in. 

The code below also shows how to  modify a ClimateParams dictionary.

In [ ]:
# Here we create the default ClimateParams dictionary
ClimateParams_default = cl.CreateClimateParams(epsdictionary_from_file)
display(ClimateParams_default)

# And here we make a modified version of it 
ClimateParams_biggerTstar = cl.CreateClimateParams(epsdictionary_from_file)
ClimateParams_biggerTstar['albedo_Tstar']=2.5
display(ClimateParams_biggerTstar)

### Code that runs meclib's version of the propagator for Cambio2
The cell below uses the meclib function **cl.run_Cambio** to perform the Euler loop, then **cl.CollectClimateTimeSeries** to extract timelines, and the  usual plt library with the label/legend to plot the results.

In [ ]:
# Running a model to get a timeline of climate variables
CS_Cambio2_list = cl.run_Cambio(cl.PropagateClimateState_Cambio2, ClimateParams_default, time, eps)

# In case you want a reminder about other variables that are available to plot
display(CS_Cambio2_list[0])

# Extract timelines of interest
time_array = cl.CollectClimateTimeSeries(CS_Cambio2_list,'time')
C_atm_array = cl.CollectClimateTimeSeries(CS_Cambio2_list,'C_atm')
C_ocean_array = cl.CollectClimateTimeSeries(CS_Cambio2_list,'C_ocean')
T_anomaly_array = cl.CollectClimateTimeSeries(CS_Cambio2_list,'T_anomaly')

# Plot carbon amounts  using the label/legend method
plt.figure()
plt.plot(time_array,C_atm_array,label='C_atm',color='brown')
plt.plot(time_array,C_ocean_array,label='C_ocean',color='blue')
plt.title('Cambio 2')
plt.xlabel('time (yr)')
plt.ylabel('Carbon amount (GtC)')
plt.grid(True)
plt.legend()

# Plot temperature anomaly using the label/legend method
plt.figure()
plt.plot(time_array,T_anomaly_array,label='T_anomaly_array',color='red')
plt.title('Cambio 2')
plt.xlabel('time (yr)')
plt.ylabel('Temp anomaly (K)')
plt.grid(True)
plt.legend()

### A plotting alternative: meclib's built-in plotting function
The cell below uses the meclib function **cl.CS_list_plots** to plot timelines. This is pretty much the same as in the cell above, just a little easier to use -- and it reports out maxima and minima, which can be handy.

In [ ]:
items_to_plot = [['C_atm','C_ocean'],'T_anomaly']
cl.CS_list_plots(CS_Cambio2_list,'Cambio2',items_to_plot)

### Comparing results of two different Cambio models
If you want to explore the difference between two Cambio models, the meclib function **cl.CS_list_compare** can be pretty handy. Below we compare a Cambio3 model with default parameters to a Cambio3 model with a bigger ice/albedo temperature threshold.

In [ ]:
# Running Cambio3 with a modified temperature anomaly
CS_Cambio3_list_default = cl.run_Cambio(cl.PropagateClimateState_Cambio3, ClimateParams_default, time, eps)
CS_Cambio3_list_biggerTstar = cl.run_Cambio(cl.PropagateClimateState_Cambio3, ClimateParams_biggerTstar, time, eps)

# Here's comparing the results
items_to_plot = ['T_anomaly']
cl.CS_list_compare([CS_Cambio3_list_default,CS_Cambio3_list_biggerTstar],['Cambio3 (default)','Cambio3 (bigger T*)'],items_to_plot)

### Tailoring your own propagator 
Some of you will find that to carry out your project idea, you'll need your own, tailored propagator. Below is an example: it's equivalent to Cambio2, but with a reduction in emissions around the time of Covid.

To carry out the Euler loop now, you just need to specify that propagator, as in



Notice that to run your tailored version of Cambio2, we need to refer to it with the prefix "cl." (as in **cl.PropagateClimateState_Cambio2**), but to run your tailored version, you just give it the name of your function (here, **PropagateClimateState_Cambio2_with_Covid**).

In [ ]:
def PropagateClimateState_Cambio2_with_Covid(previousClimateState, ClimateParams, dt, F_ha):
    """Propagates the state of the climate, with a specified anthropogenic carbon flux"""
    """Returns a new climate state"""

    # Extract constants from ClimateParams
    k_la = ClimateParams['k_la']
    k_al0 = ClimateParams['k_al0']
    k_al1 = ClimateParams['k_al1']
    k_oa = ClimateParams['k_oa']
    k_ao = ClimateParams['k_ao']
    DC = ClimateParams['DC']
    preindustrial_albedo = ClimateParams['preindustrial albedo']
    fractional_albedo_floor = ClimateParams['fractional_albedo_floor']
    albedo_Tstar = ClimateParams['albedo_Tstar']
    albedo_delta_T = ClimateParams['albedo_delta_T']
    k_al1_Tstar = ClimateParams['k_al1_Tstar']
    k_al1_deltaT = ClimateParams['k_al1_deltaT']
    fractional_k_al1_floor = ClimateParams['fractional_k_al1_floor']
    
    # Extract concentrations, albedo, etc, from the previous climate state
    C_atm = previousClimateState['C_atm']
    C_ocean = previousClimateState['C_ocean']
    albedo = previousClimateState['albedo']
    time = previousClimateState['time']
    
    # NEW: Assuming covid reduced emissions to 80% of normal (it didn't really do that much)
    if (time > 2000) & (time < 2003):
        F_ha *= 0.8

    # Get the temperature implied by the carbon in the atmosphere, and ocean pH
    T_anomaly = cl.Diagnose_T_anomaly(C_atm, preindustrial_albedo, ClimateParams)
    actual_temperature = cl.Diagnose_actual_temperature(T_anomaly)
    OceanSurfacepH = cl.Diagnose_OceanSurfacepH(C_atm,ClimateParams)
    
    # Get new fluxes
    F_la = k_la    
    F_al = k_al0 + k_al1*C_atm
    F_oa = k_oa*C_ocean    
    F_ao = k_ao*C_atm

    # Get new concentrations of carbon that depend on the fluxes
    C_atm += (F_la + F_oa - F_ao - F_al + F_ha)*dt
    C_ocean += (F_ao - F_oa)*dt
    time += dt

    # Create a new climate state with these updates
    ClimateState = makeacopy(previousClimateState)
    ClimateState['C_atm'] = C_atm
    ClimateState['C_ocean'] = C_ocean
    ClimateState['F_al'] = F_al
    ClimateState['F_la'] = F_la
    ClimateState['F_ao'] = F_ao
    ClimateState['F_oa'] = F_oa
    ClimateState['F_ha'] = F_ha
    ClimateState['F_ocean_net'] = F_oa - F_ao
    ClimateState['F_land_net'] = F_la - F_al
    ClimateState['time'] = time
    ClimateState['T_anomaly'] = T_anomaly
    ClimateState['actual temperature'] = actual_temperature
    ClimateState['OceanSurfacepH'] = OceanSurfacepH
    ClimateState['albedo'] = albedo

    # Return the new climate state
    return ClimateState

# Run the tailored version we just made (and also the meclib version of it)
CS_Cambio2_with_Covid = cl.run_Cambio(PropagateClimateState_Cambio2_with_Covid, ClimateParams_default, time, eps)
CS_Cambio2_default = cl.run_Cambio(cl.PropagateClimateState_Cambio2, ClimateParams_default, time, eps)

# Choose items to plot
items_to_plot = ['F_ha','T_anomaly']

# Plot those items
cl.CS_list_compare([CS_Cambio2_default,CS_Cambio2_with_Covid],['Cambio2','Cambio 2 with Covid'],items_to_plot)